# 01 — Exploratory Data Analysis

FIFA International Match Outcome Prediction System

---

## Setup & Imports

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Style
plt.rcParams.update({
    'figure.facecolor': '#0E1117',
    'axes.facecolor': '#1C2130',
    'axes.edgecolor': '#3D4460',
    'text.color': '#FAFAFA',
    'axes.labelcolor': '#FAFAFA',
    'xtick.color': '#FAFAFA',
    'ytick.color': '#FAFAFA',
    'grid.color': '#2D3348',
    'grid.alpha': 0.5,
})
ACCENT = '#00FF87'
PALETTE = ['#00FF87', '#FFD700', '#FF6B6B']

print('Libraries loaded ✓')


## Load Master Dataframe

In [ ]:
from src.data_pipeline import DataPipeline

# Patch save to CSV (no pyarrow needed)
import pandas as _pd
_orig = _pd.DataFrame.to_parquet
def _to_csv(self, path, **kw): self.to_csv(str(path).replace('.parquet','.csv'), index=False)
_pd.DataFrame.to_parquet = _to_csv

pipeline = DataPipeline('../data')
master = pipeline.build_master(save=True)
summary = pipeline.get_summary()

print(f"Rows        : {len(master):,}")
print(f"Columns     : {master.shape[1]}")
print(f"Date range  : {summary['date_range'][0].date()} → {summary['date_range'][1].date()}")
print(f"Teams       : {summary['n_teams']}")
print(f"Tournaments : {summary['n_tournaments']}")
master.head(3)


## Block A — Dataset Overview

In [ ]:
# Missing value analysis (missingno-style heatmap via matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Null counts bar chart
null_counts = master.isnull().sum().sort_values(ascending=False)
null_counts = null_counts[null_counts > 0]
axes[0].barh(null_counts.index, null_counts.values, color=ACCENT)
axes[0].set_title('Missing Values by Column', fontweight='bold')
axes[0].set_xlabel('Null Count')
for i, v in enumerate(null_counts.values):
    axes[0].text(v + 10, i, str(v), va='center', fontsize=9, color='#FAFAFA')

# Match count by year
master['year'] = master['date'].dt.year
yearly = master.groupby('year').size()
axes[1].bar(yearly.index, yearly.values, color=ACCENT, alpha=0.7, width=0.9)
axes[1].set_title('International Matches Per Year', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Matches')
axes[1].axvline(1945, color='#FFD700', linestyle='--', alpha=0.8, label='WWII end')
axes[1].axvline(1991, color='#FF6B6B', linestyle='--', alpha=0.8, label='USSR dissolved')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/block_a_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Total matches: {len(master):,}  |  Years: {master['year'].min()}–{master['year'].max()}")


## Block B — Match Result Distribution

In [ ]:
complete = master[master['result'].notna()].copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall distribution
result_pcts = complete['result'].value_counts(normalize=True) * 100
labels = ['Home Win', 'Draw', 'Away Win']
keys   = ['home_win', 'draw', 'away_win']
vals   = [result_pcts.get(k, 0) for k in keys]
bars = axes[0].bar(labels, vals, color=PALETTE)
axes[0].set_title('Overall Result Distribution', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold', color='#FAFAFA')

# Neutral vs Home venue
for ax, flag, title in zip(
    [axes[1], axes[2]],
    [True, False],
    ['Neutral Venue', 'Home Venue']
):
    sub = complete[complete['neutral'] == flag]
    pcts = sub['result'].value_counts(normalize=True) * 100
    v = [pcts.get(k, 0) for k in keys]
    ax.bar(labels, v, color=PALETTE)
    ax.set_title(f'Result Distribution — {title}', fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    for i, val in enumerate(v):
        ax.text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=10, color='#FAFAFA')

plt.tight_layout()
plt.savefig('../data/processed/block_b_results.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n📊 KEY INSIGHT — Home Advantage:")
neutral_hw = complete[complete['neutral']==True]['result'].eq('home_win').mean()*100
home_hw    = complete[complete['neutral']==False]['result'].eq('home_win').mean()*100
print(f"  Home win % at home venue  : {home_hw:.1f}%")
print(f"  Home win % at neutral site: {neutral_hw:.1f}%")
print(f"  Advantage differential    : {home_hw - neutral_hw:.1f} percentage points")


In [ ]:
# Home advantage by decade
complete['decade'] = (complete['year'] // 10) * 10
decade_hw = (
    complete.groupby('decade')['result']
    .apply(lambda x: (x == 'home_win').mean() * 100)
    .reset_index(name='home_win_pct')
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(decade_hw['decade'].astype(str), decade_hw['home_win_pct'],
       color=[ACCENT if v > 45 else '#FFD700' for v in decade_hw['home_win_pct']], width=0.7)
ax.axhline(50, color='#FF6B6B', linestyle='--', alpha=0.7, label='50% line')
ax.set_title('Home Win % by Decade — Is Home Advantage Shrinking?', fontweight='bold', fontsize=13)
ax.set_xlabel('Decade')
ax.set_ylabel('Home Win %')
ax.set_ylim(30, 60)
ax.legend()
for i, row in decade_hw.iterrows():
    ax.text(i, row['home_win_pct'] + 0.3, f"{row['home_win_pct']:.0f}%",
            ha='center', fontsize=8, color='#FAFAFA')
plt.tight_layout()
plt.savefig('../data/processed/block_b_home_advantage_trend.png', dpi=120, bbox_inches='tight')
plt.show()


## Block C — Goal Distribution Analysis

In [ ]:
comp = complete[complete['total_goals'].notna()].copy()
comp['total_goals'] = comp['total_goals'].astype(float)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0,0].hist(comp['total_goals'], bins=range(0, 16), color=ACCENT, edgecolor='black', alpha=0.8)
axes[0,0].set_title('Total Goals per Match Distribution', fontweight='bold')
axes[0,0].set_xlabel('Total Goals')
axes[0,0].set_ylabel('Frequency')

# Average goals by decade
decade_goals = comp.groupby('decade')['total_goals'].mean()
axes[0,1].bar(decade_goals.index.astype(str), decade_goals.values, color=PALETTE[0], width=0.7)
axes[0,1].set_title('Avg Goals per Match by Decade', fontweight='bold')
axes[0,1].set_xlabel('Decade')
axes[0,1].set_ylabel('Avg Goals')
for i, (dec, v) in enumerate(decade_goals.items()):
    axes[0,1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=8, color='#FAFAFA')

# Goals by tournament tier
tier_goals = comp.groupby('tournament_tier')['total_goals'].mean()
axes[1,0].bar(['Tier 1\n(Elite)', 'Tier 2\n(Qualifying)', 'Tier 3\n(Friendly)'],
              [tier_goals.get(1,0), tier_goals.get(2,0), tier_goals.get(3,0)],
              color=PALETTE)
axes[1,0].set_title('Avg Goals by Tournament Tier', fontweight='bold')
axes[1,0].set_ylabel('Avg Goals per Match')

# Score matrix heatmap (top 8x8)
max_score = 8
score_matrix = np.zeros((max_score+1, max_score+1))
for _, row in comp.iterrows():
    hg = min(int(row['home_goals']), max_score)
    ag = min(int(row['away_goals']), max_score)
    score_matrix[hg, ag] += 1
sns.heatmap(score_matrix, ax=axes[1,1], cmap='YlOrRd',
            xticklabels=range(max_score+1), yticklabels=range(max_score+1))
axes[1,1].set_title('Score Frequency Heatmap (Home vs Away Goals)', fontweight='bold')
axes[1,1].set_xlabel('Away Goals')
axes[1,1].set_ylabel('Home Goals')

plt.tight_layout()
plt.savefig('../data/processed/block_c_goals.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Most common scoreline: {comp.groupby(['home_goals','away_goals']).size().idxmax()}")
print(f"Average goals/match (all time): {comp['total_goals'].mean():.2f}")


## Block D — Team Performance Analysis

In [ ]:
# Build per-team all-time record
teams_data = []
all_teams = sorted(set(complete['home_team']) | set(complete['away_team']))

for team in all_teams:
    home = complete[complete['home_team'] == team]
    away = complete[complete['away_team'] == team]
    n = len(home) + len(away)
    if n == 0: continue
    w  = (home['result']=='home_win').sum() + (away['result']=='away_win').sum()
    d  = (home['result']=='draw').sum()    + (away['result']=='draw').sum()
    l  = n - w - d
    gf = home['home_goals'].fillna(0).sum() + away['away_goals'].fillna(0).sum()
    ga = home['away_goals'].fillna(0).sum() + away['home_goals'].fillna(0).sum()
    teams_data.append({'team': team, 'played': n, 'wins': w, 'draws': d,
                       'losses': l, 'gf': gf, 'ga': ga,
                       'win_pct': w/n*100 if n > 0 else 0})

teams_df = pd.DataFrame(teams_data)
qualified = teams_df[teams_df['played'] >= 50].copy()
top20 = qualified.nlargest(20, 'win_pct')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 20 by win%
axes[0].barh(top20['team'], top20['win_pct'], color=ACCENT)
axes[0].set_title('Top 20 Teams by Win % (min 50 matches)', fontweight='bold')
axes[0].set_xlabel('Win %')
axes[0].invert_yaxis()

# Top 20 by total wins
top20w = teams_df.nlargest(20, 'wins')
axes[1].barh(top20w['team'], top20w['wins'], color=PALETTE[1])
axes[1].set_title('Top 20 Teams by Total Wins', fontweight='bold')
axes[1].set_xlabel('Total Wins')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../data/processed/block_d_teams.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nTop 5 by Win%:")
print(top20[['team','played','wins','win_pct']].head().to_string(index=False))


## Block E — Temporal Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Matches per year
yearly_counts = master.groupby('year').size()
axes[0].fill_between(yearly_counts.index, yearly_counts.values, alpha=0.4, color=ACCENT)
axes[0].plot(yearly_counts.index, yearly_counts.values, color=ACCENT, linewidth=1.5)
axes[0].set_title('International Matches Played Per Year (1872–2026)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Matches')
# Annotate key events
events = {1914: 'WWI', 1939: 'WWII', 1950: 'WC resumes', 1990: 'Confederation growth'}
for yr, label in events.items():
    axes[0].axvline(yr, color='#FF6B6B', alpha=0.7, linestyle='--')
    axes[0].text(yr+0.5, yearly_counts.max()*0.9, label, fontsize=7, rotation=90, color='#FF6B6B')

# Average goals per year (rolling 5-year)
year_goals = complete[complete['total_goals'].notna()].groupby('year')['total_goals'].mean()
roll_goals = year_goals.rolling(5, min_periods=1).mean()
axes[1].plot(year_goals.index, year_goals.values, color='#555577', alpha=0.4, linewidth=1)
axes[1].plot(roll_goals.index, roll_goals.values, color=ACCENT, linewidth=2, label='5-year rolling avg')
axes[1].set_title('Average Goals Per Match by Year (5-year rolling average)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Avg Goals')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/block_e_trends.png', dpi=120, bbox_inches='tight')
plt.show()


## Block F — Penalty Shootout Analysis

In [ ]:
from src.data_pipeline import DataPipeline as DP2
p2 = DP2('../data')
p2.load_raw_data()
sht = p2.shootouts.copy()

# Shootout win rate per team
teams_in_sht = set(sht['home_team']) | set(sht['away_team'])
sht_stats = []
for team in teams_in_sht:
    appearances = sht[(sht['home_team']==team) | (sht['away_team']==team)]
    wins = (appearances['winner']==team).sum()
    n = len(appearances)
    if n >= 3:
        sht_stats.append({'team': team, 'appearances': n, 'wins': wins, 'win_pct': wins/n*100})

sht_df = pd.DataFrame(sht_stats).sort_values('win_pct', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top shootout teams
top_sht = sht_df.head(15)
axes[0].barh(top_sht['team'], top_sht['win_pct'], color=ACCENT)
axes[0].set_title('Shootout Win % (min 3 appearances)', fontweight='bold')
axes[0].set_xlabel('Win %')
axes[0].invert_yaxis()
axes[0].axvline(50, color='#FF6B6B', linestyle='--', alpha=0.7)

# First shooter advantage
has_first = sht[sht['first_shooter'].notna()].copy()
if len(has_first) > 0:
    first_wins = (has_first['first_shooter'] == has_first['winner']).mean() * 100
    second_wins = 100 - first_wins
    axes[1].bar(['Shoots First', 'Shoots Second'], [first_wins, second_wins],
                color=[ACCENT, '#FF6B6B'])
    axes[1].set_title(f'First-Shooter Advantage\n(n={len(has_first)} shootouts with data)',
                      fontweight='bold')
    axes[1].set_ylabel('Win %')
    axes[1].axhline(50, color='white', linestyle='--', alpha=0.5)
    for i, v in enumerate([first_wins, second_wins]):
        axes[1].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold', color='#FAFAFA')
    print(f"\n📊 First-shooter wins: {first_wins:.1f}%  (of {len(has_first)} recorded shootouts)")
    print(f"   Note: {sht['first_shooter'].isna().sum()} shootouts have no first_shooter data")
else:
    axes[1].text(0.5, 0.5, 'No first_shooter data available', ha='center', va='center',
                 transform=axes[1].transAxes, color='#FAFAFA')

plt.tight_layout()
plt.savefig('../data/processed/block_f_shootouts.png', dpi=120, bbox_inches='tight')
plt.show()


## Summary — Key Analytical Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS FROM EDA")
print("=" * 60)
print()

comp2 = master[master['result'].notna()].copy()
home_hw  = comp2[comp2['neutral']==False]['result'].eq('home_win').mean()*100
neut_hw  = comp2[comp2['neutral']==True]['result'].eq('home_win').mean()*100
oldest   = comp2.groupby('decade')['result'].apply(lambda x: (x=='home_win').mean()*100)

print(f"1. HOME ADVANTAGE: Home venue HW={home_hw:.1f}% vs Neutral={neut_hw:.1f}%")
print(f"   1880s={oldest.get(1880,0):.0f}% → 2020s={oldest.get(2020,0):.0f}% (trend: {'↓ shrinking' if oldest.get(2020,0) < oldest.get(1880,0) else '↑ growing'})")
print()

draws_t1 = comp2[comp2['tournament_tier']==1]['result'].eq('draw').mean()*100
draws_t3 = comp2[comp2['tournament_tier']==3]['result'].eq('draw').mean()*100
print(f"2. DRAWS: Elite matches={draws_t1:.1f}% draws vs Friendlies={draws_t3:.1f}% draws")
print()

goals_t1 = comp2[comp2['tournament_tier']==1]['total_goals'].mean()
goals_t3 = comp2[comp2['tournament_tier']==3]['total_goals'].mean()
print(f"3. GOALS: Elite avg={goals_t1:.2f} goals vs Friendly avg={goals_t3:.2f} goals/match")
print()

modern_goals = comp2[comp2['year']>=2010]['total_goals'].mean()
historic_goals = comp2[comp2['year']<1970]['total_goals'].mean()
print(f"4. GOAL TRENDS: Pre-1970={historic_goals:.2f} vs Post-2010={modern_goals:.2f} goals/match")
print()

top5 = teams_df.nlargest(5, 'win_pct')[['team','played','win_pct']]
print(f"5. DOMINANT TEAMS:")
for _, row in top5.iterrows():
    print(f"   {row['team']}: {row['win_pct']:.1f}% win rate ({row['played']:.0f} matches)")
